## setup

In [1]:
import os, cv2, shutil, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from typing import Tuple, Union
from collections import defaultdict

In [2]:
F_DATA = "./data/all-heart"
F_DATA_ORIGINAL = F_DATA + "/original"
F_DATA_ORIGINAL_IMAGES = F_DATA_ORIGINAL + "/images"
F_DATA_ORIGINAL_MASKS = F_DATA_ORIGINAL + "/masks"
F_DATA_AUGMENTED = F_DATA + "/augmented"
F_DATA_AUGMENTED_IMAGES = F_DATA_AUGMENTED + "/images"
F_DATA_AUGMENTED_MASKS = F_DATA_AUGMENTED + "/masks"

CSV_PATH_ORIGINAL = F_DATA + "/original_data.csv"
CSV_PATH_AUGMENTED = F_DATA + "/augmented_data.csv"

In [15]:
class DataAugmenter:
    def __init__(self, csv_path, f_original_images, f_original_masks, f_augmented_images, f_augmented_masks, output_csv_path):
        self.csv_path = csv_path
        self.output_csv_path = output_csv_path
        # folder paths
        self.f_original_images = f_original_images
        self.f_original_masks = f_original_masks
        self.f_augmented_images = f_augmented_images
        self.f_augmented_masks = f_augmented_masks

    def _empty_folder(self, folder):
        if os.path.exists(folder):
            shutil.rmtree(folder)
        os.makedirs(folder)

    def _crop_padding(self, img):
        if img is None:
            return None, None
        
        h, w = img.shape
        mean_rows = np.mean(img, axis=1)
        mean_cols = np.mean(img, axis=0)
        low_thresh, high_thresh = 15, 240

        rows_to_keep = (mean_rows >= low_thresh) & (mean_rows <= high_thresh)
        if np.any(rows_to_keep):
            first_row = np.argmax(rows_to_keep)
            last_row = len(rows_to_keep) - np.argmax(rows_to_keep[::-1]) - 1
        else:
            first_row, last_row = 0, h - 1
        
        cols_to_keep = (mean_cols >= low_thresh) & (mean_cols <= high_thresh)
        if np.any(cols_to_keep):
            first_col = np.argmax(cols_to_keep)
            last_col = len(cols_to_keep) - np.argmax(cols_to_keep[::-1]) - 1
        else:
            first_col, last_col = 0, w - 1
        
        crop_coords = (first_row, last_row, first_col, last_col)
        
        if last_row >= first_row and last_col >= first_col:
            cropped = img[first_row:last_row+1, first_col:last_col+1]
            return cropped, crop_coords
        else:
            return img, crop_coords

    def _restore_padding(self, cropped_img, crop_coords, original_shape):
        first_row, last_row, first_col, last_col = crop_coords
        h, w = original_shape
        
        restored = np.zeros((h, w), dtype=cropped_img.dtype)
        restored[first_row:last_row+1, first_col:last_col+1] = cropped_img
        
        return restored

    def _adjust_exposure_to_target(self, img, target_brightness, max_iter=50, tol=0.5):
        current_brightness = img.mean()
        
        # Handle extreme cases
        if target_brightness <= 0:
            beta = -255.0
        elif target_brightness >= 255:
            beta = 255.0
        else:
            # Bisection search for optimal beta
            low, high = -255.0, 255.0
            beta = 0.0
            
            for _ in range(max_iter):
                beta = (low + high) / 2.0
                adjusted = np.clip(img.astype(np.float32) + beta, 0, 255)
                new_brightness = adjusted.mean()
                
                if abs(new_brightness - target_brightness) < tol:
                    break
                    
                if new_brightness < target_brightness:
                    low = beta
                else:
                    high = beta
        
        # Final adjustment
        adjusted = np.clip(img.astype(np.float32) + beta, 0, 255).astype(np.uint8)
        return adjusted

    def _adjust_black_to_target(self, img, target_brightness, max_iter=50, tol=0.5):
        current_brightness = img.mean()
        
        # Handle extreme cases
        if target_brightness <= 0:
            value = -255.0
        elif target_brightness >= 255:
            value = 255.0
        else:
            # Bisection search for optimal value
            low, high = -255.0, 255.0
            value = 0.0
            
            for _ in range(max_iter):
                value = (low + high) / 2.0
                
                # Build and apply LUT
                lut = np.arange(256, dtype=np.float32)
                scale = 1 - (lut / 255)
                adjusted_lut = np.clip(lut + value * scale, 0, 255).astype(np.uint8)
                adjusted = cv2.LUT(img, adjusted_lut)
                
                new_brightness = adjusted.mean()
                
                if abs(new_brightness - target_brightness) < tol:
                    break
                    
                if new_brightness < target_brightness:
                    low = value
                else:
                    high = value
        
        # Final adjustment
        lut = np.arange(256, dtype=np.float32)
        scale = 1 - (lut / 255)
        adjusted_lut = np.clip(lut + value * scale, 0, 255).astype(np.uint8)
        adjusted = cv2.LUT(img, adjusted_lut)
        return adjusted

    def _find_image_path(self, name, folder):
        for ext in ['.png', '.jpg', '.jpeg']:
            path = os.path.join(folder, name + ext)
            if os.path.exists(path):
                return path
        return None

    def _get_file_extension(self, filepath):
        return os.path.splitext(filepath)[1]

    def _create_ctr_groups(self, ctr_group_gap):
        groups = []
        # Calculate how many groups we need around 0.5
        start = 0.5
        
        # Groups going down from 0.5
        current = start
        while current - ctr_group_gap >= 0:
            current -= ctr_group_gap
            groups.insert(0, (current, current + ctr_group_gap))
        
        # Groups starting from 0.5
        current = start
        while current < 1.0:
            if current + ctr_group_gap <= 1.0:
                groups.append((current, current + ctr_group_gap))
            current += ctr_group_gap
        
        return groups

    def _create_brightness_groups(self, brightness_group_gap):
        groups = []
        current = 0
        while current < 255:
            end = min(current + brightness_group_gap, 255)
            groups.append((current, end))
            current = end
        return groups

    def _get_group_index(self, value, groups):
        for i, (start, end) in enumerate(groups):
            if start <= value < end:
                return i
        return -1  # Not in any group

    def augment_by_adjusting_brightness(self, copy_amount_dict, brightness_shift_limit, min_brightness_shift_limit, ctr_group_gap, brightness_group_gap, valid_brightness_range):
        # Load CSV data
        df = pd.read_csv(self.csv_path)
        
        # Empty output folders
        self._empty_folder(self.f_augmented_images)
        self._empty_folder(self.f_augmented_masks)
        
        # Create groups
        ctr_groups = self._create_ctr_groups(ctr_group_gap)
        brightness_groups = self._create_brightness_groups(brightness_group_gap)
        
        # Calculate current brightness for all images (on cropped versions)
        current_brightness = []
        for _, row in df.iterrows():
            img_path = self._find_image_path(row['name'], self.f_original_images)
            if img_path:
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                cropped_img, _ = self._crop_padding(img)
                current_brightness.append(cropped_img.mean())
            else:
                current_brightness.append(row['brightness'])  # fallback to CSV value
        
        df['original_brightness'] = current_brightness
        
        # Group images by CTR
        ctr_group_counts = defaultdict(int)
        for _, row in df.iterrows():
            ctr_group_idx = self._get_group_index(row['ctr'], ctr_groups)
            if ctr_group_idx >= 0:
                ctr_group_counts[ctr_group_idx] += 1
        
        # Track brightness group counts (including original data)
        brightness_group_counts = defaultdict(int)
        for brightness in current_brightness:
            brightness_group_idx = self._get_group_index(brightness, brightness_groups)
            if brightness_group_idx >= 0:
                brightness_group_counts[brightness_group_idx] += 1
        
        # Prepare output data
        output_data = []
        
        # Copy original data
        for _, row in df.iterrows():
            img_path = self._find_image_path(row['name'], self.f_original_images)
            mask_path = self._find_image_path(row['name'], self.f_original_masks)
            
            if img_path and mask_path:
                # Copy original files
                img_ext = self._get_file_extension(img_path)
                mask_ext = self._get_file_extension(mask_path)
                
                shutil.copy2(img_path, os.path.join(self.f_augmented_images, row['name'] + img_ext))
                shutil.copy2(mask_path, os.path.join(self.f_augmented_masks, row['name'] + mask_ext))
                
                # Copy all columns from original row, update brightness
                row_data = row.to_dict()
                row_data['brightness'] = row['original_brightness']
                output_data.append(row_data)
        
        # Process augmentation
        for _, row in df.iterrows():
            ctr_group_idx = self._get_group_index(row['ctr'], ctr_groups)
            if ctr_group_idx < 0:
                continue  # Skip if not in any CTR group
            
            # Determine number of copies based on CTR group count
            ctr_count = ctr_group_counts[ctr_group_idx]
            copies_to_make = 0
            
            for num_copies, (min_count, max_count) in copy_amount_dict.items():
                if min_count <= ctr_count <= max_count:
                    copies_to_make = num_copies
                    break
            
            if copies_to_make == 0:
                continue
            
            # Load original image and mask
            img_path = self._find_image_path(row['name'], self.f_original_images)
            mask_path = self._find_image_path(row['name'], self.f_original_masks)
            
            if not img_path or not mask_path:
                continue
            
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            original_shape = img.shape
            
            img_ext = self._get_file_extension(img_path)
            mask_ext = self._get_file_extension(mask_path)
            
            # Crop image to calculate brightness range
            cropped_img, crop_coords = self._crop_padding(img)
            original_brightness = cropped_img.mean()
            
            # Determine valid brightness range
            min_brightness = max(valid_brightness_range[0], original_brightness - brightness_shift_limit)
            max_brightness = min(valid_brightness_range[1], original_brightness + brightness_shift_limit)
            
            # Find available brightness groups within range and minimum shift
            available_groups = []
            for i, (start, end) in enumerate(brightness_groups):
                group_center = (start + end) / 2
                # Check if group is within valid range, shift limits and meets minimum shift requirement
                if (start >= min_brightness and end <= max_brightness and 
                    start >= valid_brightness_range[0] and end <= valid_brightness_range[1] and
                    abs(group_center - original_brightness) >= min_brightness_shift_limit):
                    available_groups.append(i)
            
            if not available_groups:
                continue
            
            # Create copies
            for copy_idx in range(copies_to_make):
                # Sort available groups by current count (ascending)
                available_groups.sort(key=lambda x: brightness_group_counts[x])
                
                # Choose the group with lowest count
                target_group_idx = available_groups[copy_idx % len(available_groups)]
                group_start, group_end = brightness_groups[target_group_idx]
                
                # Choose random target brightness within the group range and valid brightness range
                target_brightness = np.random.uniform(
                    max(group_start, valid_brightness_range[0]), 
                    min(group_end, valid_brightness_range[1])
                )
                
                # Adjust brightness
                if target_brightness > original_brightness:
                    adjusted_cropped = self._adjust_exposure_to_target(cropped_img, target_brightness)
                else:
                    adjusted_cropped = self._adjust_black_to_target(cropped_img, target_brightness)
                
                # Restore padding
                adjusted_img = self._restore_padding(adjusted_cropped, crop_coords, original_shape)
                
                # Save augmented image and mask
                aug_name = f"{row['name']}-aug{copy_idx + 1}"
                aug_img_path = os.path.join(self.f_augmented_images, aug_name + img_ext)
                aug_mask_path = os.path.join(self.f_augmented_masks, aug_name + mask_ext)
                
                cv2.imwrite(aug_img_path, adjusted_img)
                cv2.imwrite(aug_mask_path, mask)  # Mask remains unchanged
                
                # Update brightness group count
                brightness_group_counts[target_group_idx] += 1
                
                # Add to output data
                actual_brightness = adjusted_cropped.mean()
                
                # Copy all columns from original row, update name and brightness
                row_data = row.to_dict()
                row_data['name'] = aug_name
                row_data['brightness'] = actual_brightness
                output_data.append(row_data)
        
        # Save output CSV
        output_df = pd.DataFrame(output_data)
        output_df.to_csv(self.output_csv_path, index=False)
        
        print(f"Augmentation completed. Total images: {len(output_df)}")
        print(f"Original images: {len(df)}")
        print(f"Augmented images: {len(output_df) - len(df)}")

In [16]:
data_augmenter = DataAugmenter(CSV_PATH_ORIGINAL, F_DATA_ORIGINAL_IMAGES, F_DATA_ORIGINAL_MASKS, F_DATA_AUGMENTED_IMAGES, F_DATA_AUGMENTED_MASKS, CSV_PATH_AUGMENTED)

## augment data by adjusting brightness

In [17]:
copy_amount_dict = {
    7: (1,5),
    5: (6,10),
    2: (11,15),
    1: (16,20)
}
brightness_shift_limit = 50
min_brightness_shift_limit = 20
ctr_group_gap = 0.01
brightness_group_gap = 5
valid_brightness_range = (91, 224)

In [18]:
data_augmenter.augment_by_adjusting_brightness(
    copy_amount_dict=copy_amount_dict,
    brightness_shift_limit=brightness_shift_limit,
    min_brightness_shift_limit=min_brightness_shift_limit,
    ctr_group_gap=ctr_group_gap,
    brightness_group_gap=brightness_group_gap,
    valid_brightness_range=valid_brightness_range
    )

Augmentation completed. Total images: 1025
Original images: 383
Augmented images: 642
